# X0(N) Isogeny Construction

Interactive front-end for `modular_isogeny_lib.sage`.

This notebook computes the rational maps for paper coefficients `a_4, a_6, a_4', a_6'` and summarizes isogeny analysis at rational points of `X_0(N)`.


In [ ]:
import html as html_lib
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML, Math
from time import perf_counter
from sage.all import *
# 1. Load Library
load("modular_isogeny_lib.sage")
# ==============================================================================
# Main Execution Logic
# ==============================================================================

def run_isogeny_construction(N, prec=500, allow_fixed_plan_fallback=False):
    """Run the algebraic construction pipeline for X0(N).
    Output convention (paper normalization):
      a_4  = u,   a_6  = v,
      a_4' = u_N, a_6' = v_N.
    """
    clear_output()
    print(f"Starting computation for N = {N} with precision {prec}...")
    t_start = perf_counter()
    try:
        model_data = get_model_data(N)
    except NotImplementedError as e:
        print(f"\n[Error] {e}")
        return
    P_xy = PolynomialRing(QQ, names=['x', 'y'])
    print("Step 1: Computing Modular Forms...")
    print("Step 2: Retrieving X0(N) Model Generators...")
    context_cache = {}
    def get_context(work_prec):
        p = int(work_prec)
        if p not in context_cache:
            forms_p = setup_modular_forms(N, p)
            x_p, y_p = model_data['generators'](forms_p['q'])
            context_cache[p] = {
                'forms': forms_p,
                'x0': x_p,
                'y0': y_p,
                'solver_cache_by_deg_y': {}
            }
        return context_cache[p]
    try:
        full_ctx = get_context(prec)
    except NotImplementedError as e:
        print(f"\n[Error] {e}")
        return
    forms = full_ctx['forms']
    x0_full = full_ctx['x0']
    y0_full = full_ctx['y0']
    print("Step 3: Solving for Rational Maps (Adaptive Degree + Adaptive Precision)...")
    degree_search_plan = {}
    if N == 43:
        degree_search_plan = {
            'u':   {'start': 8, 'min': 8, 'max': 8, 'step': 1, 'max_deg_y': 3},
            'u_N': {'start': 8, 'min': 8, 'max': 8, 'step': 1, 'max_deg_y': 3},
            'v':   {'start': 12, 'min': 12, 'max': 12, 'step': 1, 'max_deg_y': 3},
            'v_N': {'start': 12, 'min': 12, 'max': 12, 'step': 1, 'max_deg_y': 3},
        }
    elif N == 67:
        degree_search_plan = {
            'u':   {'start': 15, 'min': 15, 'max': 15, 'step': 1, 'max_deg_y': 6},
            'u_N': {'start': 15, 'min': 15, 'max': 15, 'step': 1, 'max_deg_y': 6},
            'v':   {'start': 19, 'min': 19, 'max': 19, 'step': 1, 'max_deg_y': 6},
            'v_N': {'start': 19, 'min': 19, 'max': 19, 'step': 1, 'max_deg_y': 6},
        }
    elif N == 163:
        degree_search_plan = {
            'u':   {'start': 34, 'min': 34, 'max': 34, 'step': 1, 'max_deg_y': 11},
            'u_N': {'start': 34, 'min': 34, 'max': 34, 'step': 1, 'max_deg_y': 11},
            'v':   {'start': 38, 'min': 38, 'max': 38, 'step': 1, 'max_deg_y': 12},
            'v_N': {'start': 38, 'min': 38, 'max': 38, 'step': 1, 'max_deg_y': 12},
        }
    base_start_deg = 10
    if degree_search_plan:
        base_start_deg = min(plan['start'] for plan in degree_search_plan.values())
    # Empirical default: dynamic y-bound helps some levels but can slow others.
    use_dynamic_max_deg_y = (N in [11, 43])
    solver_L0_margin = 5
    solver_Delta = 30
    solver_verification_order = 100
    verify_order_full_prec = min(max(solver_verification_order + 10, 110), max(50, prec - 5))
    if N == 163:
        # For level 163, weak full-precision checks can admit spurious near-solutions.
        verify_order_full_prec = max(verify_order_full_prec, min(max(250, prec // 2), max(80, prec - 5)))
    # Precision adaptation knobs
    prec_min_floor = max(120, solver_verification_order + 10)
    prec_est_safety = 20
    prec_quant_step = 20
    prec_adaptive_deg_cap = 30 if N >= 67 else 25
    use_dual_precision_consensus = (N <= 27)
    dual_prec_min_gap = 30
    try:
        val_x0 = int(x0_full.valuation())
    except Exception:
        val_x0 = -2
    try:
        val_y0 = int(y0_full.valuation())
    except Exception:
        val_y0 = -3
    target_valuations = {}
    for key in ['f', 'fN', 'g', 'gN']:
        try:
            target_valuations[key] = int(forms[key].valuation())
        except Exception:
            target_valuations[key] = 0
    curve_y_degree = None
    try:
        x_tmp, y_tmp = P_xy.gens()
        curve_y_degree = model_data['poly_xy'](x_tmp, y_tmp).degree(y_tmp)
        if curve_y_degree <= 0:
            curve_y_degree = None
    except Exception:
        curve_y_degree = None
    def get_solver_cache_for_context(ctx, max_deg_y):
        key = int(max_deg_y)
        cache_by_deg_y = ctx['solver_cache_by_deg_y']
        if key not in cache_by_deg_y:
            cache_by_deg_y[key] = {}
        return cache_by_deg_y[key]
    def get_max_deg_y_candidates(total_deg):
        default_max_deg_y = max(1, total_deg // 3)
        if (not use_dynamic_max_deg_y) or curve_y_degree is None or curve_y_degree < 2:
            return [default_max_deg_y]
        tight_max_deg_y = min(default_max_deg_y, curve_y_degree - 1)
        if tight_max_deg_y == default_max_deg_y:
            return [default_max_deg_y]
        return [tight_max_deg_y, default_max_deg_y]
    def monomial_count(deg, max_deg_y):
        j_max = min(deg, max_deg_y)
        return (j_max + 1) * (deg + 1) - (j_max * (j_max + 1)) // 2
    def min_monomial_valuation(deg, max_deg_y):
        j_max = min(deg, max_deg_y)
        best_val = None
        for j in range(j_max + 1):
            i = (deg - j) if val_x0 < 0 else 0
            v = i * val_x0 + j * val_y0
            if best_val is None or v < best_val:
                best_val = v
        return int(best_val if best_val is not None else 0)
    def estimate_required_prec(target_key, total_deg, max_deg_y):
        req = prec_min_floor
        target_val = target_valuations.get(target_key, 0)
        for deg_B_total in range(total_deg + 1):
            deg_A_total = total_deg - deg_B_total
            k_A = monomial_count(deg_A_total, max_deg_y)
            k_B = monomial_count(deg_B_total, max_deg_y)
            L0 = k_A + k_B + solver_L0_margin
            k_min_A = min_monomial_valuation(deg_A_total, max_deg_y)
            k_min_B = min_monomial_valuation(deg_B_total, max_deg_y) + target_val
            k_min = min(k_min_A, k_min_B)
            req_split = k_min + L0 + max(0, solver_Delta) + prec_est_safety
            if req_split > req:
                req = req_split
        req = int(max(req, prec_min_floor))
        req = ((req + prec_quant_step - 1) // prec_quant_step) * prec_quant_step
        req = max(min(prec, req), min(prec, prec_min_floor))
        return req
    def get_attempt_precisions(target_key, total_deg, max_deg_y):
        if not use_dual_precision_consensus:
            return [prec]
        if total_deg > prec_adaptive_deg_cap:
            return [prec]
        est_prec = estimate_required_prec(target_key, total_deg, max_deg_y)
        if est_prec >= prec:
            return [prec]
        if (prec - est_prec) < dual_prec_min_gap:
            return [prec]
        return [est_prec, prec]
    def verify_rat_on_full_precision(rat, target_key):
        if rat is None:
            return False
        try:
            S = rat(x=x0_full, y=y0_full) - forms[target_key]
            return S.valuation() >= verify_order_full_prec
        except Exception:
            return False
    def rats_agree_on_full_precision(rat_a, rat_b):
        if rat_a is None or rat_b is None:
            return False
        try:
            D = rat_a(x=x0_full, y=y0_full) - rat_b(x=x0_full, y=y0_full)
            return D.valuation() >= verify_order_full_prec
        except Exception:
            return False
    # --------------------------------------------------------------------------
    # Helper: Adaptive Search Function
    # --------------------------------------------------------------------------
    def find_map_adaptive(target_key, label, start_deg=None, max_deg=100, step=5, min_deg_floor=1, fixed_max_deg_y=None):
        """
        Tries to find the rational map. If failed, increases degree and retries.
        Returns (rational_map, minimal_total_degree).
        """
        if start_deg is None:
            start_deg = base_start_deg
        start_deg = max(base_start_deg, start_deg)
        if start_deg % step != 0:
            start_deg += step - (start_deg % step)
        min_deg_floor = max(1, int(min_deg_floor))
        explored_upto = min_deg_floor - 1
        tight_pref_enabled = True
        for deg in range(start_deg, max_deg + 1, step):
            min_deg = max(min_deg_floor, explored_upto + 1)
            if fixed_max_deg_y is None:
                all_deg_y_candidates = get_max_deg_y_candidates(deg)
            else:
                all_deg_y_candidates = [int(fixed_max_deg_y)]
            if (not tight_pref_enabled) and len(all_deg_y_candidates) >= 2:
                deg_y_candidates = [all_deg_y_candidates[-1]]
            else:
                deg_y_candidates = all_deg_y_candidates
            tight_was_tried = (
                len(all_deg_y_candidates) >= 2
                and len(deg_y_candidates) >= 1
                and deg_y_candidates[0] == all_deg_y_candidates[0]
            )
            tight_failed = False
            for idx, max_deg_y in enumerate(deg_y_candidates):
                attempt_precs = get_attempt_precisions(target_key, deg, max_deg_y)
                highest_prec = attempt_precs[-1]
                provisional_low = None
                for work_prec in attempt_precs:
                    print(
                        f"  - Trying {label} with degree window=[{min_deg}, {deg}], "
                        f"max_deg_y={max_deg_y}, prec={work_prec}...",
                        end="\r"
                    )
                    ctx = get_context(work_prec)
                    target_f = ctx['forms'][target_key]
                    x0 = ctx['x0']
                    y0 = ctx['y0']
                    local_cache = get_solver_cache_for_context(ctx, max_deg_y)
                    sols = find_minimal_robust_solution(
                        target_f, x0, y0, P_xy,
                        max_total_degree=deg,
                        min_total_degree=min_deg,
                        max_deg_y=max_deg_y,
                        L0_margin=solver_L0_margin,
                        Delta=solver_Delta,
                        verification_order=solver_verification_order,
                        verbose=False,
                        use_cache=True,
                        precomputed=local_cache,
                        profile=False,
                        include_q_series=False
                    )
                    if sols:
                        cand = sols[0]
                        rat_candidate = AB_ratio(cand)
                        if not verify_rat_on_full_precision(rat_candidate, target_key):
                            if work_prec < prec:
                                print(
                                    f"  -> Rejected {label} candidate at prec={work_prec} "
                                    f"(failed full-precision verification)."
                                )
                            continue
                        if use_dual_precision_consensus and len(attempt_precs) >= 2 and work_prec < highest_prec:
                            provisional_low = (rat_candidate, cand['total_degree'], work_prec)
                            print(
                                f"  -> Provisional {label} at prec={work_prec}; "
                                f"confirming at prec={highest_prec}."
                            )
                            continue
                        if provisional_low is not None:
                            low_rat, _, low_prec = provisional_low
                            if rats_agree_on_full_precision(low_rat, rat_candidate):
                                print(
                                    f"  -> Dual-precision consensus for {label}: "
                                    f"prec={low_prec} and prec={work_prec} agree."
                                )
                            else:
                                print(
                                    f"  -> Dual-precision mismatch for {label}: "
                                    f"using high-precision candidate from prec={work_prec}."
                                )
                        print(
                            f"  -> Found {label} "
                            f"(max_degree={deg}, max_deg_y={max_deg_y}, prec={work_prec})"
                        )
                        return rat_candidate, cand['total_degree']
                if provisional_low is not None:
                    _, _, low_prec = provisional_low
                    print(
                        f"  -> Discarded provisional {label} from prec={low_prec} "
                        f"(no confirmed high-precision solution at this split)."
                    )
                if tight_was_tried and idx == 0:
                    tight_failed = True
            if tight_was_tried and tight_failed:
                tight_pref_enabled = False
            explored_upto = deg
            print(f"  -> Could not find {label} within degree {deg}.")
        print(f"  -> Failed to find {label} even at max_degree={max_deg}")
        return None, None
    # --------------------------------------------------------------------------
    # Execute Search
    # --------------------------------------------------------------------------
    search_order = [("u", 'f'), ("u_N", 'fN'), ("v", 'g'), ("v_N", 'gN')]
    found_maps = {}
    found_degrees = {}
    next_start_deg = base_start_deg
    for label, target_key in search_order:
        plan = degree_search_plan.get(label, None)
        if plan is None:
            rat, found_deg = find_map_adaptive(
                target_key, label,
                start_deg=next_start_deg
            )
        else:
            print(
                f"  -> Using fixed-degree plan for {label}: "
                f"min={plan.get('min', plan['start'])}, start={plan['start']}, "
                f"max={plan['max']}, step={plan['step']}, max_deg_y={plan.get('max_deg_y', 'auto')}"
            )
            rat, found_deg = find_map_adaptive(
                target_key, label,
                start_deg=plan['start'],
                max_deg=plan['max'],
                step=plan['step'],
                min_deg_floor=plan.get('min', plan['start']),
                fixed_max_deg_y=plan.get('max_deg_y', None)
            )
            if rat is None:
                if allow_fixed_plan_fallback:
                    print(
                        f"  -> Fixed-degree plan failed for {label}; "
                        f"fallback to adaptive search from degree {plan['start']}."
                    )
                    rat, found_deg = find_map_adaptive(
                        target_key, label,
                        start_deg=plan['start'],
                        max_deg=100,
                        step=1,
                        min_deg_floor=plan['start']
                    )
                else:
                    print(
                        f"  -> Fixed-degree plan failed for {label}. "
                        f"Set allow_fixed_plan_fallback=True to run full adaptive fallback."
                    )
        found_maps[label] = rat
        found_degrees[label] = found_deg
        if found_deg is not None:
            print(f"  -> {label} minimal total degree: {found_deg}")
            next_start_deg = found_deg
    u = found_maps['u']
    v = found_maps['v']
    uN = found_maps['u_N']
    vN = found_maps['v_N']
    if not all([u, v, uN, vN]):
        missing = [lbl for lbl in ['u', 'u_N', 'v', 'v_N'] if found_maps.get(lbl) is None]
        missing_str = ', '.join(missing) if missing else 'unknown'
        clear_output()
        display(HTML(
            f"<div style='font-family:-apple-system, BlinkMacSystemFont, Segoe UI, sans-serif;'>"
            f"<h3>Computation Incomplete (N={N})</h3>"
            f"<p>Could not determine: <span style='font-family:ui-monospace, SFMono-Regular, Menlo, monospace;'>{html_lib.escape(missing_str)}</span>.</p>"
            f"<p>Try increasing precision or set <span style='font-family:ui-monospace, SFMono-Regular, Menlo, monospace;'>allow_fixed_plan_fallback=True</span>.</p>"
            f"</div>"
        ))
        return
    a4, a6, a4N, a6N = get_paper_coeff_maps(u, v, uN, vN)
    formula_map = build_latex_formula_map(
        u, v, uN, vN, model_data['poly_xy'], include_raw=False
    )
    if ('a4' not in formula_map) or (not formula_map['a4']):
        formula_map['a4'] = latex(a4)
    if ('a4N' not in formula_map) or (not formula_map['a4N']):
        formula_map['a4N'] = latex(a4N)
    if ('a6' not in formula_map) or (not formula_map['a6']):
        formula_map['a6'] = latex(a6)
    if ('a6N' not in formula_map) or (not formula_map['a6N']):
        formula_map['a6N'] = latex(a6N)
    analysis_data = analyze_isogenies(
        u, v, uN, vN, N, model_data,
        return_data=True,
        emit_text=False
    )
    analysis_global = analysis_data.get('global_lines', []) if analysis_data else []
    point_blocks = analysis_data.get('points', []) if analysis_data else []
    elapsed = perf_counter() - t_start
    point_cards = []
    for blk in point_blocks:
        domain_items = blk.get('domain_raw', []) if blk else []
        codomain_items = blk.get('codomain_raw', []) if blk else []
        domain_items = domain_items if domain_items else ['n/a']
        codomain_items = codomain_items if codomain_items else ['n/a']
        count = blk.get('isogeny_count', None)
        if count is None:
            count = extract_isogeny_count(domain_items + codomain_items, N)
        non_isogeny = blk.get('non_isogeny_flag', None)
        if non_isogeny is None:
            non_isogeny = has_non_isogeny_flag(domain_items + codomain_items)
        if (count is not None and count > 0) and not non_isogeny:
            card_bg = '#ecfdf3'
            card_border = '#8ed3a7'
            badge_bg = '#d6f5e1'
            badge_text = '#146c43'
            status_label = f'{N}-isogeny exists (count = {count})'
        else:
            card_bg = '#fff1f2'
            card_border = '#f3a0a7'
            badge_bg = '#ffdfe2'
            badge_text = '#9f1239'
            status_label = f'No {N}-isogeny detected'
            if count is not None:
                status_label = f'{N}-isogeny count = {count}'
        point_cards.append({
            'point_raw': blk.get('point_raw', blk.get('point', '-')),
            'a_pair': blk.get('a_pair', '-'),
            'a_pairN': blk.get('a_pairN', '-'),
            'domain_raw': list(domain_items),
            'codomain_raw': list(codomain_items),
            'domain_section': blk.get('domain_section', None),
            'codomain_section': blk.get('codomain_section', None),
            'card_bg': card_bg,
            'card_border': card_border,
            'badge_bg': badge_bg,
            'badge_text': badge_text,
            'status': status_label,
        })
    header_html = (
        "<div style='font-family:-apple-system, BlinkMacSystemFont, Segoe UI, sans-serif; line-height:1.45;'>"
        f"<h2 style='margin:0;'>X_0({N}) Isogeny Construction</h2>"
        f"<p style='margin:6px 0 10px 0; color:#444;'>Completed in {elapsed:.1f}s with precision q^{prec}.</p>"
        "<div style='padding:10px 12px; border:1px solid #ddd; border-radius:8px; background:#fafafa;'>"
        "<b>Displayed outputs</b>: paper coefficients a_4, a_4', a_6, a_6' only."
        "</div></div>"
    )
    clear_output()
    display(HTML(header_html))
    display(HTML("<h3 style='margin-top:16px;'>Recovered Formulae</h3>"))
    display(Math(f"a_4(X,Y) = {formula_map['a4']}"))
    display(Math(f"a_4'(X,Y) = {formula_map['a4N']}"))
    display(Math(f"a_6(X,Y) = {formula_map['a6']}"))
    display(Math(f"a_6'(X,Y) = {formula_map['a6N']}"))
    if analysis_global:
        analysis_items = []
        for item in analysis_global:
            s_item = str(item).strip()
            if not s_item:
                continue
            if s_item.startswith('=== Analysis of Cyclic'):
                continue
            analysis_items.append(s_item)
        if analysis_items:
            chips_html = ''.join(
                f"<span style='display:inline-block; padding:5px 9px; border-radius:999px; background:#f4f7fb; border:1px solid #d9e2ee; color:#2d3a4a; font-size:12px;'>"
                f"{html_lib.escape(x)}"
                f"</span>"
                for x in analysis_items
            )
            display(HTML("<h3 style='margin-top:16px;'>Analysis Summary</h3>"))
            display(HTML(
                "<div style='border:1px solid #d9d9d9; border-radius:10px; background:#fff; padding:10px 12px;'>"
                "<div style='display:flex; flex-wrap:wrap; gap:8px;'>"
                f"{chips_html}"
                "</div></div>"
            ))
    if point_cards:
        display(HTML("<h3 style='margin-top:16px;'>Isogeny Analysis by Rational Point</h3>"))
        for card in point_cards:
            card_layout = widgets.Layout(
                border=f"1px solid {card['card_border']}",
                padding='10px 12px',
                margin='10px 0',
            )
            try:
                card_layout.background_color = card['card_bg']
            except Exception:
                pass
            card_box = widgets.Output(layout=card_layout)
            with card_box:
                display(HTML(
                    "<div style='display:flex; align-items:center; justify-content:space-between; gap:8px; margin-bottom:6px;'>"
                    f"<div style='font-weight:600;'>P = <span style='font-family:ui-monospace, SFMono-Regular, Menlo, monospace;'>{html_lib.escape(card['point_raw'])}</span></div>"
                    f"<div style='font-size:12px; padding:2px 8px; border-radius:12px; background:{card['badge_bg']}; color:{card['badge_text']};'>{html_lib.escape(card['status'])}</div>"
                    "</div>"
                ))
                domain_section = card.get('domain_section', None)
                if domain_section is None:
                    domain_section = parse_curve_section(card['domain_raw'], card['a_pair'], r"(a_4,\ a_6)", N)
                codomain_section = card.get('codomain_section', None)
                if codomain_section is None:
                    codomain_section = parse_curve_section(card['codomain_raw'], card['a_pairN'], r"(a_4^{\prime},\ a_6^{\prime})", N)
                render_domain_codomain_tables(domain_section, codomain_section, indent_px=8, math_in_cells=False)
            display(card_box)
# ==============================================================================

# UI Widget Setup
# ==============================================================================
n_selector = widgets.Dropdown(
    options=[11, 14, 15, 17,19,21,27,37,43,67,163],
    value=11,
    description='Select N:',
    disabled=False,
)
run_button = widgets.Button(
    description='Run Computation',
    button_style='success',
    tooltip='Click to start calculation',
    icon='check'
)
out = widgets.Output()
def on_button_clicked(b):
    with out:
        clear_output()
        try:
            run_isogeny_construction(n_selector.value)
        except Exception as e:
            print(f"An error occurred: {e}")
            import traceback
            traceback.print_exc()
run_button._click_handlers.callbacks = []
run_button.on_click(on_button_clicked)
print("Select N and click Run:")
display(n_selector, run_button, out)
